[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/01.%20Clase%201/01Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F01.%20Clase%201%2F01Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 1: Análisis de Acciones con Python

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

---

## Introducción teórica

En los mercados financieros, el **precio ajustado de cierre** $S_t^{\text{adj}}$ corrige el precio histórico por *splits* y dividendos, permitiendo comparaciones válidas a lo largo del tiempo.

Una serie de precios $\{S_t\}_{t=0}^{T}$ es una **serie de tiempo financiera**: una secuencia de observaciones indexadas por el tiempo de negociación. Su análisis estadístico es el punto de partida para la gestión de portafolios y la medición de riesgo.

### Hipótesis de Mercados Eficientes (EMH)

La **EMH** (Fama, 1970) postula que los precios reflejan toda la información disponible. Se distinguen tres formas:

| Forma | Información incorporada | Implicación |
|-------|------------------------|-------------|
| **Débil** | Precios e historial de transacciones | El análisis técnico no genera rendimientos extraordinarios |
| **Semi-fuerte** | Toda la información pública | El análisis fundamental tampoco basta |
| **Fuerte** | Información pública y privada | Ni los *insiders* pueden obtener ventaja |

Si la forma débil se cumple, los precios pasados por sí solos no predicen rendimientos futuros — lo que motiva el análisis estadístico que desarrollaremos en este notebook.

## 1. Descarga de datos históricos con `yfinance`

Usamos el paquete `yfinance` para descargar precios ajustados directamente de Yahoo! Finance.  
Consideramos tres acciones (Alcoa, Apple y Microsoft) y el índice S&P 500.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scipy.stats as stats

# Estilo visual
sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 8)

In [ ]:
tickers     = ["AA", "AAPL", "MSFT", "^GSPC"]
start_date  = "2014-01-01"
end_date    = "2016-12-31"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
print(f"Filas: {len(data)}  |  Columnas: {list(data.columns.get_level_values(0).unique())}")
data.head()

`yfinance` devuelve un `DataFrame` con columnas de dos niveles:  
- **Nivel 0**: campo (`Close`, `Open`, `High`, `Low`, `Volume`)  
- **Nivel 1**: ticker  

Acceder a todos los precios de cierre es directo:

In [ ]:
closes = data["Close"]
closes.head()

In [ ]:
# Precio de cierre de Microsoft
closes["MSFT"].head()

In [ ]:
# Precio de cierre en una fecha específica
closes.loc["2014-01-14"]

In [ ]:
# Todos los campos de un ticker
data.xs("^GSPC", level=1, axis=1).head()

---
## 2. Visualización de precios

### 2.1 Precio ajustado de cierre

In [ ]:
fig, ax = plt.subplots()
closes[["AA", "AAPL", "MSFT"]].plot(ax=ax)
ax.set_title("Precio ajustado de cierre (2014-2016)")
ax.set_ylabel("USD")
plt.tight_layout()

La escala del S&P 500 es muy diferente a la de las acciones individuales,  
por lo que se grafican por separado.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
closes[["AA", "AAPL", "MSFT"]].plot(ax=axes[0], title="Acciones individuales")
axes[0].set_ylabel("USD")
closes["^GSPC"].plot(ax=axes[1], title="S&P 500", color="steelblue")
axes[1].set_ylabel("Puntos")
plt.tight_layout()

### 2.2 Precio y volumen de Microsoft

In [ ]:
msftA = closes["MSFT"]
msftV = data["Volume"]["MSFT"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)
ax1.plot(msftA.index, msftA, label="Precio ajustado")
ax1.set_title("Microsoft: Precio ajustado de cierre 2014-2016")
ax1.set_ylabel("USD")
ax1.legend()

ax2.bar(msftV.index, msftV, width=1, color="steelblue", alpha=0.7)
ax2.set_title("Volumen diario")
ax2.set_ylabel("Volumen")
plt.tight_layout()

### 2.3 Medias móviles

### Nota teórica: medias móviles como filtros

La **media móvil simple** (SMA) de ventana $n$ actúa como un filtro pasa-bajas que suaviza el ruido de alta frecuencia:

$$
\text{SMA}_t(n) = \frac{1}{n}\sum_{i=0}^{n-1} S_{t-i}
$$

En análisis técnico, el cruce de una SMA corta (e.g., 20 días) por encima de una SMA larga (e.g., 100 días) se denomina **golden cross** (señal alcista), y el cruce inverso, **death cross** (señal bajista).

> **Conexión con la EMH**: bajo la hipótesis de caminata aleatoria $S_t = S_{t-1} + \varepsilon_t$, los cruces de medias no tienen poder predictivo. Cualquier aparente "señal" es un artefacto estadístico del suavizado (Luenberger, 2013).

Las **medias móviles** suavizan la serie de precios y ayudan a identificar tendencias.  
Usamos ventanas de 20 y 100 días.

In [ ]:
sma20  = msftA.rolling(20).mean()
sma100 = msftA.rolling(100).mean()

fig, ax = plt.subplots()
ax.plot(msftA,  label="Precio ajustado",       alpha=0.6)
ax.plot(sma20,  label="Media móvil 20 días",   linewidth=2)
ax.plot(sma100, label="Media móvil 100 días",  linewidth=2)
ax.set_title("Microsoft: Precio y medias móviles")
ax.set_ylabel("USD")
ax.legend()
plt.tight_layout()

### 2.4 Bandas de volatilidad (desviación móvil)

### Nota teórica: Bandas de Bollinger

Las **Bandas de Bollinger** (Bollinger, 2002) envuelven el precio con una banda superior e inferior definidas por:

$$
\text{Upper}_t = \text{SMA}_t(n) + k \cdot \sigma_t(n), \qquad \text{Lower}_t = \text{SMA}_t(n) - k \cdot \sigma_t(n)
$$

donde $\sigma_t(n)$ es la desviación estándar móvil de ventana $n$ y típicamente $k = 2$.

Bajo el supuesto de normalidad, con $k=2$ la banda contiene aproximadamente el 95 % de las observaciones. En el código siguiente usamos $k=1$ (≈ 68 %), lo que permite visualizar períodos de alta y baja volatilidad relativa.

In [ ]:
std20  = msftA.rolling(20).std()
std100 = msftA.rolling(100).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, std, window in zip(axes, [std20, std100], [20, 100]):
    ax.plot(msftA, label="Precio ajustado", alpha=0.7, color="steelblue")
    ax.fill_between(msftA.index,
                    msftA - std, msftA + std,
                    alpha=0.25, color="orange",
                    label=f"± 1 std ({window} días)")
    ax.set_ylabel("USD")
    ax.legend()
    ax.set_title(f"Banda de volatilidad — ventana {window} días")
plt.tight_layout()

### Nota teórica: fundamentos de los rendimientos financieros

El modelo estándar para la dinámica de precios de activos es el **Movimiento Browniano Geométrico** (GBM):

$$
dS_t = \mu \, S_t \, dt + \sigma \, S_t \, dW_t
$$

donde $\mu$ es la tasa de rendimiento esperado (drift), $\sigma$ es la volatilidad y $W_t$ es un proceso de Wiener estándar. Este modelo es la base del modelo de Black-Scholes-Merton para valoración de opciones (Hull, 2018).

Aplicando el **lema de Itô** a $\ln S_t$, se obtiene que los **rendimientos logarítmicos** se distribuyen normalmente:

$$
r_t = \ln S_t - \ln S_{t-1} \sim \mathcal{N}\!\left(\left(\mu - \tfrac{\sigma^2}{2}\right)\Delta t,\; \sigma^2 \Delta t\right)
$$

**Ventajas del rendimiento logarítmico** sobre el simple:
1. **Aditividad temporal**: $r_{t_0 \to t_n} = \sum_{i=1}^{n} r_{t_{i-1} \to t_i}$
2. **Simetría**: una ganancia de $+x$ y una pérdida de $-x$ son simétricas
3. **Consistencia estadística**: bajo GBM, los log-retornos son i.i.d. normales (Tsay, 2010)

---
## 3. Rendimientos

### 3.1 Rendimiento simple

Para una serie de precios $\{S_t\}$, el **rendimiento simple** es:
$$
R_t = \frac{S_t - S_{t-1}}{S_{t-1}}
$$

In [ ]:
stocks = closes[["AA", "AAPL", "MSFT"]]
R = stocks.pct_change().dropna()
R.head()

In [ ]:
fig, ax = plt.subplots()
R.plot(ax=ax, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Rendimientos simples diarios (2014-2016)")
ax.set_ylabel("Rendimiento")
plt.tight_layout()

### 3.2 Rendimiento logarítmico

El **rendimiento logarítmico** (continuamente compuesto) se define como:
$$
r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right) = \ln(1 + R_t)
$$
Para rendimientos pequeños, $r_t \approx R_t$.  
El rendimiento log tiene la ventaja de ser **aditivo** en el tiempo.

In [ ]:
r = np.log(stocks / stocks.shift(1)).dropna()
r.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
R.plot(ax=axes[0], alpha=0.7, title="Rendimiento simple")
r.plot(ax=axes[1], alpha=0.7, title="Rendimiento logarítmico")
for ax in axes:
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Rendimiento")
plt.tight_layout()

### 3.3 Estadísticas descriptivas

In [ ]:
R.describe()

In [ ]:
pd.DataFrame({
    "Asimetría (skewness)": r.skew(),
    "Curtosis (exceso)": r.kurtosis()
})

### Nota teórica: volatilidad como medida de riesgo

La **volatilidad anualizada** se estima a partir de la desviación estándar de los rendimientos diarios:

$$
\hat{\sigma}_{\text{anual}} = \hat{\sigma}_{\text{diario}} \times \sqrt{252}
$$

El factor $\sqrt{252}$ proviene del supuesto de que los rendimientos son **independientes e idénticamente distribuidos** (i.i.d.), y que hay 252 días hábiles por año. Bajo este supuesto, la varianza escala linealmente con el tiempo y la desviación estándar escala con la raíz cuadrada.

> **Limitación**: en la práctica, los rendimientos financieros exhiben **clustering de volatilidad** — períodos de alta volatilidad tienden a ser seguidos por más alta volatilidad. Este fenómeno, conocido como **efecto ARCH** (Engle, 1982), viola el supuesto i.i.d. y motiva los modelos GARCH que se estudiarán más adelante en el curso (Tsay, 2010, Cap. 3).

In [ ]:
r.describe()

### Nota teórica: rendimiento acumulado y composición continua

El valor de una inversión inicial de $1 USD al cabo de $T$ períodos se expresa de dos formas equivalentes:

$$
\frac{S_T}{S_0} = \prod_{t=1}^{T}(1 + R_t) = \exp\!\left(\sum_{t=1}^{T} r_t\right)
$$

La primera usa rendimientos simples (composición discreta); la segunda usa rendimientos logarítmicos (composición continua). La equivalencia se debe a que $r_t = \ln(1 + R_t)$.

Esta propiedad permite que el rendimiento acumulado logarítmico sea simplemente la **suma** de los rendimientos diarios, simplificando el análisis estadístico (Luenberger, 2013, Cap. 2).

### 3.4 Volatilidad móvil

La **volatilidad** anualizada se estima como la desviación estándar de los rendimientos multiplicada por $\sqrt{252}$.

In [ ]:
vol_R = R.rolling(20).std() * np.sqrt(252)
vol_r = r.rolling(20).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
vol_R.plot(ax=axes[0], title="Volatilidad anualizada — rendimiento simple (ventana 20 días)")
vol_r.plot(ax=axes[1], title="Volatilidad anualizada — rendimiento log (ventana 20 días)")
for ax in axes:
    ax.set_ylabel("Volatilidad")
plt.tight_layout()

### Nota teórica: normalidad y colas pesadas

El supuesto de **normalidad** de los rendimientos es central en modelos como Black-Scholes, el VaR paramétrico y el CAPM. Sin embargo, en la práctica los rendimientos financieros exhiben desviaciones sistemáticas (McNeil, Frey & Embrechts, 2015):

- **Leptocurtosis** (colas pesadas): la curtosis en exceso $\kappa > 0$ indica más observaciones extremas de lo que predice la normal
- **Asimetría negativa**: las caídas grandes son más frecuentes que las subidas equivalentes

Formalmente, para una variable aleatoria $r$ con media $\mu$ y desviación estándar $\sigma$:

$$
\text{Skew}[r] = \mathbb{E}\!\left[\left(\frac{r - \mu}{\sigma}\right)^{\!3}\right], \qquad \text{Kurt}[r] = \mathbb{E}\!\left[\left(\frac{r - \mu}{\sigma}\right)^{\!4}\right] - 3
$$

Para una distribución normal, $\text{Skew} = 0$ y $\text{Kurt} = 0$. El **test de Jarque-Bera** evalúa conjuntamente si la asimetría y curtosis son compatibles con la normal:

$$
JB = \frac{n}{6}\left(\text{Skew}^2 + \frac{\text{Kurt}^2}{4}\right) \sim \chi^2(2)
$$

Los **QQ-plots** ofrecen una evaluación visual: si los puntos se desvían de la diagonal en los extremos, hay evidencia de colas pesadas (Tsay, 2010, Cap. 1).

El **rendimiento acumulado** muestra el valor de $1 USD invertido al inicio del período.

In [ ]:
cum_R = (1 + R).cumprod()
cum_r = np.exp(r.cumsum())

fig, ax = plt.subplots()
cum_R.plot(ax=ax)
ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Rendimiento acumulado (base = 1 USD, 2014-2016)")
ax.set_ylabel("Valor de la inversión")
plt.tight_layout()

---
## 4. Distribución de los rendimientos

#### Test de Jarque-Bera

Aplicamos el test formal de normalidad. Un **p-valor < 0.05** rechaza la hipótesis nula de normalidad.

In [ ]:
### Nota teórica: correlación y diversificación

La **varianza de un portafolio** de dos activos con pesos $w_1$ y $w_2$ es:

$$
\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2\,w_1\,w_2\,\rho_{12}\,\sigma_1\,\sigma_2
$$

Cuando el coeficiente de correlación $\rho_{12} < 1$, la varianza del portafolio es **menor** que el promedio ponderado de las varianzas individuales. Este es el **principio fundamental de la diversificación** (Markowitz, 1952).

La **matriz de dispersión** y el **mapa de calor de correlación** que calculamos a continuación permiten visualizar la estructura de co-movimiento entre activos. Activos con baja correlación son mejores candidatos para diversificación — tema central de las Clases 2 a 8.

Analizamos si los rendimientos siguen una distribución normal.  
Esta hipótesis es central en muchos modelos financieros (Black-Scholes, CAPM, etc.).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, r.columns):
    r[col].hist(bins=50, ax=ax, edgecolor="white")
    ax.set_title(f"Histograma — {col}")
    ax.set_xlabel("Rendimiento log")
plt.suptitle("Distribución de rendimientos logarítmicos")
plt.tight_layout()

**QQ-Plot**: si los puntos caen sobre la línea roja, la distribución es aproximadamente normal.

---

## 5. Referencias bibliográficas

- **Bollinger, J.** (2002). *Bollinger on Bollinger Bands*. McGraw-Hill.
- **Engle, R. F.** (1982). Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation. *Econometrica*, 50(4), 987–1007.
- **Fama, E. F.** (1970). Efficient Capital Markets: A Review of Theory and Empirical Work. *The Journal of Finance*, 25(2), 383–417.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 15: The Black-Scholes-Merton Model.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 2: Basic Fixed-Income Securities; Cap. 6–8: Mean-Variance Portfolio Theory.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **McNeil, A. J., Frey, R. & Embrechts, P.** (2015). *Quantitative Risk Management: Concepts, Techniques and Tools* (2nd ed.). Princeton University Press. — Cap. 3: Empirical Properties of Financial Data.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. — Cap. 1: Financial Time Series and Their Characteristics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, r.columns):
    stats.probplot(r[col].dropna(), dist="norm", plot=ax)
    ax.set_title(f"QQ-Plot — {col}")
plt.suptitle("Prueba de normalidad (QQ-Plot)")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots()
r.plot(kind="box", ax=ax)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Boxplot de rendimientos logarítmicos")
ax.set_ylabel("Rendimiento log")
plt.tight_layout()

La **matriz de dispersión** muestra la relación entre los rendimientos de cada par de activos.  
Una correlación positiva alta implica que los activos se mueven juntos (menor diversificación).

In [ ]:
pd.plotting.scatter_matrix(r, diagonal="kde", alpha=0.3, figsize=(9, 9))
plt.suptitle("Matriz de dispersión de rendimientos logarítmicos", y=1.01)
plt.tight_layout()

### Matriz de correlación

In [ ]:
corr = r.corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlación de rendimientos logarítmicos")
plt.tight_layout()